In [0]:
import time
from datetime import datetime
from pyspark.sql import SparkSession
from pyspark.sql.types import *

class DatabricksObservability:

    def __init__(self, catalog_name, environment="dev"):
        self.catalog = catalog_name
        self.environment = environment
        self.execution_id = None
        self.start_time = None
        self.spark = SparkSession.builder.getOrCreate()

    def start_execution(self, job_name):
        self.execution_id = f"{job_name}_{int(time.time())}"
        self.start_time = datetime.now()
        return self.execution_id

    def log_ingestion_metric(
        self,
        source_table,
        ingestion_type,
        start_time,
        end_time,
        records_loaded,
        landing_path,
        status="success"
        # glue_job_name=None,
        # glue_job_run_id=None,
        # error_message=None
    ):
        try:
            # glue_job_name = glue_job_name or ""
            # glue_job_run_id = glue_job_run_id or ""
            # error_message = error_message or ""

            duration = (end_time - start_time).total_seconds()

            data = [(
                self.execution_id,
                "landing",
                source_table,
                ingestion_type,
                self.environment,
                start_time,
                end_time,
                duration,
                records_loaded,
                records_loaded,
                landing_path,
                # glue_job_name,
                # glue_job_run_id,
                status,
                # error_message,
                datetime.now()
            )]

            columns = [
                "execution_id","source_system","source_table","ingestion_type","environment",
                "start_time","end_time","duration_seconds","records_extracted","records_loaded",
                "landing_path","status","created_at" #,"error_message""glue_job_name","glue_job_run_id",
            ]

            df = self.spark.createDataFrame(data, columns)
            df.write.mode("append").saveAsTable(f"{self.catalog}.observability.ingestion_metrics")

        except Exception as e:
            print(f"Erro ingestion: {e}")

    def log_job_execution(
        self,
        job_name,
        layer,
        start_time,
        end_time,
        status,
        records_processed,
        records_failed=0,
        source_system="landing",
        error_message=None,
        metadata=None
    ):
        try:

            import json

            metadata = json.dumps(metadata) if metadata else ""
            error_message = error_message or ""
            duration = (end_time - start_time).total_seconds()

            data = [(
                self.execution_id,
                job_name,
                layer,
                self.environment,
                start_time,
                end_time,
                duration,
                status,
                records_processed,
                records_failed,
                source_system,
                error_message,
                #metadata,
                datetime.now()
            )]

            columns = [
                "execution_id","job_name","layer","environment","start_time","end_time",
                "duration_seconds","status","records_processed","records_failed",
                "source_system","error_message","created_at" #,"metadata"
            ]

            df = self.spark.createDataFrame(data, columns)
            df.write.mode("append").saveAsTable(f"{self.catalog}.observability.job_execution_metrics")

        except Exception as e:
            print(f"Erro job_execution: {e}")

    def log_data_quality(
        self,
        table_name,
        layer,
        check_type,
        check_name,
        status,
        records_checked,
        records_passed,
        records_failed=0,
        threshold=100.0,
        error_details=None
    ):
        try:
            pass_rate = (records_passed / records_checked * 100) if records_checked > 0 else 0
            check_id = f"{self.execution_id}_{table_name}_{int(time.time())}"

            data = [(
                check_id,
                self.execution_id,
                table_name,
                layer,
                check_type,
                check_name,
                status,
                records_checked,
                records_passed,
                records_failed,
                pass_rate,
                threshold,
                error_details,
                datetime.now()
            )]

            columns = [
                "check_id","execution_id","table_name","layer","check_type","check_name",
                "status","records_checked","records_passed","records_failed",
                "pass_rate","threshold","error_details","created_at"
            ]

            df = self.spark.createDataFrame(data, columns)
            df.write.mode("append").saveAsTable(f"{self.catalog}.observability.data_quality_metrics")

        except Exception as e:
            print(f"Erro data_quality: {e}")

In [0]:
#Parâmetros

#dbutils.widgets.text("input_path", "")
#dbutils.widgets.text("table_name", "")
dbutils.widgets.text("layer", "bronze")

layer = dbutils.widgets.get("layer")

files = dbutils.fs.ls("/Volumes/bikestore/resource/origem/")

In [0]:
layer = "bronze"

In [0]:



# input_path = dbutils.widgets.get("input_path")
# table_name = dbutils.widgets.get("table_name")
#layer = dbutils.widgets.get("layer")

# Running with parameters:
# input_path = {input_path}
# table_name = {table_name}
print(f"""
layer = {layer}
""")

In [0]:


# spark.sql(f"USE CATALOG {catalog})

In [0]:
from pyspark.sql.functions import current_timestamp, col

# observabilidade
obs = DatabricksObservability("bikestore", "dev")
#obs.start_execution(f"{layer}_{table_name}")

from datetime import datetime
job_start = datetime.now()

for file in files:

    input_path = file.path

    table_name = (
        file.name
        .replace(".csv", "")
    )

    print(f"""
    Processing:
    input_path = {input_path}
    table_name = {table_name}
    layer = {layer}
    """)

    obs.start_execution(f"{layer}_{table_name}")

    job_start = datetime.now()

    try:

        df = (
        spark.read
        .format("csv")
        .option("header", True)
        .option("inferSchema", True)
        .load(input_path)
        .withColumn("bronze_ingest_timestamp", current_timestamp())
        .withColumn("bronze_source_file", col("_metadata.file_path"))
        )


        count = df.count()

        df.write \
            .mode("overwrite") \
            .option("overwriteSchema", "true") \
            .saveAsTable(
                f"{obs.catalog}.{layer}.{table_name}"
            )

        job_end = datetime.now()

        # ingestion
        obs.log_ingestion_metric(
            source_table=table_name,
            ingestion_type="batch",
            start_time=job_start,
            end_time=job_end,
            records_loaded=count,
            landing_path=input_path,
            status="success"
        )

        # execution
        obs.log_job_execution(
            job_name=table_name,
            layer=layer,
            start_time=job_start,
            end_time=job_end,
            status="success",
            records_processed=count
        )

        print(f"{table_name} processada")

    except Exception as e:

        job_end = datetime.now()

        obs.log_job_execution(
            job_name=table_name,
            layer=layer,
            start_time=job_start,
            end_time=job_end,
            status="failed",
            records_processed=0,
            error_message=str(e)
        )

        print(f"Erro em {table_name}: {e}")

In [0]:
# from pyspark.sql.functions import col
# # # validação
# # if not input_path:
# #     raise ValueError("input_path não foi informado")

# # if not table_name:
# #     raise ValueError("table_name não foi informado")



# # observabilidade
# obs = DatabricksObservability("bikestore", "dev")
# obs.start_execution(f"{layer}_{table_name}")

# from datetime import datetime
# job_start = datetime.now()

# try:
#     df = spark.read.csv(input_path, header=True)

#     # força tudo para string
#     #df = df.select([col(c).cast("string") for c in df.columns])
#     count = 0
#     count = df.count()

#     df.write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(f"{obs.catalog}.{layer}.{table_name}") #####

#     job_end = datetime.now()

#     # ingestão
#     obs.log_ingestion_metric(
#         source_table=table_name,
#         ingestion_type="batch",
#         start_time=job_start,
#         end_time=job_end,
#         records_loaded=count,
#         landing_path=input_path,
#         status="success"
#     )

#     # execução
#     obs.log_job_execution(
#         job_name=table_name,
#         layer=layer,
#         start_time=job_start,
#         end_time=job_end,
#         status="success",
#         records_processed=count,
#         # metadata={
#         #     "input_path": input_path
#         # }
#     )

# except Exception as e:
#     job_end = datetime.now()

#     obs.log_job_execution(
#         job_name=table_name,
#         layer=layer,
#         start_time=job_start,
#         end_time=job_end,
#         status="success",
#         records_processed=count
# )

#     raise e